##### Boilerplate

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"
NOTEBOOKS_DIR = os.path.join(PROJECT_DIR, "notebooks")
SRC_DIR = os.path.join(PROJECT_DIR, "src")

os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, SRC_DIR)
sys.path.insert(0, NOTEBOOKS_DIR)

print("DumLLM initialized!")
print("Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DumLLM initialized!
Working directory: /content/drive/MyDrive/DumLLM


# 7. Fine-tuning to follow instructions

In [6]:
from importlib.metadata import version

pkgs = ["matplotlib",  # Plotting library
        "numpy",       # PyTorch & TensorFlow dependency
        "tiktoken",    # Tokenizer
        "torch",       # Deep learning library
        "tensorflow",  # For OpenAI's pretrained weights
        "pandas"       # Dataset loading
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

matplotlib version: 3.10.0
numpy version: 2.4.6
tiktoken version: 0.14.0
torch version: 2.14.1+cpu
tensorflow version: 2.21.0
pandas version: 2.2.3


## 7.2 Preparing a dataset for supervised instruction finetuning

In [8]:
import json
import os
import urllib
from pathlib import Path


def load_file(file_path):
  with open(file_path, "rb") as f:
    data = json.loads(f.read().decode())
  return data

file_path = Path("public/instruction-data.json")
data = load_file(file_path)

In [15]:
print("Number of  examples:", len(data))
print(data[50])

Number of  examples: 1100
{'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [17]:
def format_input(entry):
  instruction_text = (
      f"Below is an instruction that describes a task. "
      "Write a response that appropriately completes the request.\n\n"
      f"### Instruction:\n{entry['instruction']}"
  )

  input_text = f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""

  return instruction_text + input_text

In [19]:
print(format_input(data[50]))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion


In [20]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


In [24]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_potion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [25]:
print("Train data size:", len(train_data))
print("Test data size:", len(test_data))
print("Validation data size:", len(val_data))

Train data size: 935
Test data size: 110
Validation data size: 55
